# 06 - Feature Selection & Multicollinearity Analysis

## 1. Objectives
- Assess multicollinearity using Variance Inflation Factor (VIF).
- Evaluate feature importance using baseline tree models.
- Determine the optimal feature subset to prevent noise and overfitting.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features.feature_engineering import LoanFeatureEngineer
from src.features.feature_selection import compute_vif, select_features

df = pd.read_csv(project_root / "data" / "interim" / "cleaned_bank_loan.csv")
engineer = LoanFeatureEngineer(drop_identifiers=True)
df_features = engineer.fit_transform(df)

X = df_features.drop(columns=["Personal Loan"])
y = df_features["Personal Loan"]

## 2. Multicollinearity Assessment via VIF

In [2]:
numeric_cols = ["Age", "Experience", "Income", "CCAvg", "Mortgage", "Family"]
vif_df = compute_vif(X, numeric_cols)
print("Variance Inflation Factor (VIF):")
display(vif_df)

Variance Inflation Factor (VIF):


,feature,VIF
0,Age,87.281067
1,Experience,87.269772
2,Income,1.810096
3,CCAvg,1.722094
4,Mortgage,1.045892
5,Family,1.031056


As expected, `Age` and `Experience` exhibit high VIF due to their natural life correlation. Tree ensembles (RF, XGBoost, CatBoost) are naturally robust to collinearity and can partition them independently.

## 3. Feature Importance Selection via Random Forest

In [3]:
selected_cols, imp_df = select_features(X, y, importance_threshold=0.005)
print(f"Selected {len(selected_cols)} / {len(X.columns)} features:")
display(imp_df)

2026-10-26 17:56:13 [INFO] feature_selection (feature_selection.py:95): Selected 14 / 18 features above importance threshold 0.0050


Selected 14 / 18 features:


,feature,importance
2,Income,0.366487
4,CCAvg,0.131268
11,Income_Per_Family,0.125207
5,Education,0.102683
17,High_Networth_Indicator,0.066571
3,Family,0.054538
12,CCAvg_to_Income_Ratio,0.048536
15,Total_Accounts,0.020883
8,CD Account,0.019072
16,Experience_to_Age_Ratio,0.012827
